In [1]:
#We will use the mpmath library, which is used for real and complex calculations with high precision.
from mpmath import mp

## Roots of the plastic equation

The Padovan numbers are governed by the characteristic equation below. Its unique real root $p \approx 1.3247$ is the *plastic number*, and the other two roots are a pair of complex conjugates $q, \bar q$ with $|q| < p$. This gap between the moduli of the roots is what makes the Padovan numbers behave like $a\,p^m$ up to an exponentially small relative error.

Roots of the plastic equation:

$$x^3-x-1 = 0$$

In the next cell we compute the three roots with 50 digits of precision, and we use the real root $p$ to define the static density $\rho$ used throughout the article.

In [2]:
#We use the mpmath.polyroots command to obtain the roots of the polynomial with high precision.
mp.dps = 50  # We set the precision to 50 digits
roots = mp.polyroots([ 1, 0, -1, -1])  # Coefficients of the polynomial

print("Roots of the polynomial with high precision:")
for root in roots:
    print(root)
    
    
#We assign the names used in the article to the variables so we can use them in the computations.
p = roots[0]
q = roots[1]
r = roots[2]

rho = mp.fdiv(mp.power(p,3), mp.fadd(mp.fmul(2,p),3))

Roots of the polynomial with high precision:
1.3247179572447460259609088544780973407344040569017
(-0.66235897862237301298045442723904867036720202845087 - 0.56227951206230124389918214490937306149784300289578j)
(-0.66235897862237301298045442723904867036720202845087 + 0.56227951206230124389918214490937306149784300289578j)


## Binet-type formula and the error constants $\kappa$ and $\theta$

The Padovan numbers can be written through a Binet-type formula in terms of the three roots of the plastic equation. The real root $p$ gives the dominant term $a\,p^m$, while the two complex roots $q,\bar q$ contribute only the small correction $\epsilon_m$. We know that:

$$P_m = ap^m (1+\epsilon_m), \quad a = \frac{p^5}{2p+3} \quad \epsilon_m = \frac{c_q q^m + \bar{c_q q^m}}{ap^m}$$

Taking absolute values in the expression of $\epsilon_m$, the modulus of the complex part is controlled by the ratio $|q|/p<1$. Thus, for the relative error we obtain an exponentially decaying bound:

$$|\epsilon_m |\leq \frac{2|c_q|}{a}\left( |\frac{q}{p}|\right)^m $$

With the coefficient

$$c_q = \frac{1}{3q^2 -1}$$

In the next cell we compute $a$, $c_q$ and the two constants $\kappa = 2|c_q|/a$ and $\theta = |q|/p$, so that $|\epsilon_m|\le \kappa\,\theta^m$. The commented-out lines at the end of that cell are an optional test of this bound for small values of $m$.

In [3]:
a = mp.fdiv(mp.power(p, 5), mp.fadd(mp.fmul(2, p), 3))
c_q =  mp.power(q,4) / (3*mp.power(q,2) - 1)
kappa = mp.fdiv(mp.fmul(2,mp.fabs(c_q)),a)
theta = mp.fdiv(mp.fabs(q),p)

print("\nValues of the variables:")
print(f"a = {a}")
print(f"c_q = {c_q}")
print(f"kappa = {kappa}")
print(f"theta = {theta}")


#def P(m):
#    P0,P1,P2 = 1,1,1
#    seq=[P0,P1,P2]
#    for i in range(3,m+1): seq.append(seq[i-2]+seq[i-3])
#    return seq[m]
#for m in range(0,30):
#    eps = P(m)/(a*p**m) - 1
#    print(m, abs(eps) <= kappa*theta**m)


Values of the variables:
a = 0.72212441830311284114380309256468687756507175570919
c_q = (0.13893779084844357942809845371765656121746412214541 + 0.20225012409895253966317438501397276126883906919356j)
kappa = 0.67959164600284858325406315422413405635705915531656
theta = 0.65586561809714246835177399307044933349921456503319


## Claim 1: bound for the local boundary deviation $g_{n,j}$

The goal of this claim is to show that the quantity $\frac{1}{\pi_{n,j}}$ is exponentially close to the macroscopic value $\frac1\rho$, with an error that splits into a left-boundary part (decaying like $\theta^{j-1}$) and a right-boundary part (decaying like $\theta^{n-j}$). The constants $2.044$ and $3.704$ are the ones verified in the cells below.

Claim 1:

$n \geq 6,  1\leq j \leq \lfloor \frac{n}{2} \rfloor$

$$|g_{n-j}|=|\frac{P_{n+1}}{P_{j-1}P_{n-j}} - \frac{1}{\rho}| \leq 2.044 \theta^{j-1} + 3.704 \theta^{n-j}$$

### Left-boundary term

We first isolate the part of the deviation that only involves the left boundary, that is, the quantity $d_j$ comparing $\frac{p^{j+1}}{P_{j-1}}$ with $\frac1\rho$. Using the Binet-type formula of the previous section, this difference is controlled by $\kappa\theta^{j-1}$.

First, note that:

$$d_j =\frac{p^{j+1}}{P_{j-1}} - \frac{1}{\rho} $$
$$|d_j| \leq \frac{1}{\rho} \frac{\kappa \theta^{j-1}}{1-\kappa \theta^{j-1}}$$
$$Si, \quad j \geq 3, |d_j| \leq \frac{1}{\rho} \frac{\kappa \theta^{j-1}}{1-\kappa \theta^{3}}$$

For $j\ge3$ the denominator can be bounded uniformly, which gives the general bound. The small cases $j=1,2,3$ (that is, the values involving $P_0$, $P_1$ and $P_2$) are checked directly in the next cell, comparing $d_j$ with $2.044\,\theta^{j-1}$.

In [4]:
#Verification of the cases P_0, P_1 and P_2

d_0 =  mp.fsub(mp.power(p,1),mp.fdiv(1,rho))
bound_0 = mp.fmul(2.044, mp.power(theta,-1))
d_1 = mp.fsub(mp.power(p,2),mp.fdiv(1,rho))
bound_1 = mp.fmul(2.044, mp.power(theta,0))
d_2 = mp.fsub(mp.power(p,3),mp.fdiv(1,rho))
bound_2 = mp.fmul(2.044, mp.power(theta,1))
d_3 = mp.fsub(mp.power(p,4), mp.fdiv(1,rho))


print(mp.fsub(bound_2, d_2))
print(mp.fsub(bound_1, d_1))
print(mp.fsub(bound_0, d_0))
print(mp.fsub(mp.fmul(2.044, mp.power(theta,2)), d_3))


#General case (bound)

bound_left_side = mp.fdiv(kappa, mp.fmul(rho, mp.fsub(1,mp.fmul(kappa,mp.power(theta,3)))))
print(bound_left_side)

#Verification that the constant that is added p^{j+1}/P_{j-1} is less than p^4
b= mp.fdiv(   mp.fadd(mp.fmul(2,p),3)   ,  mp.power(p,3)  )
c = mp.power(p,4)

print(mp.fsub(c,b))

1.4460310751477599390698475107936300334172956973001
2.7192820427552540131189416123274128821774306598951
4.2219338251338177859164950660561186332091602690079
0.22981053071048708936585788128805751446804842122867
2.0432758117876833087375559118235081579472992721673
0.64943591448949205192181770895619468146880811380347


### Right-boundary term

Next we compare $\frac{1}{\pi_{n,j}}$ with $\frac{p^{j+1}}{P_{j-1}}$. The only difference between them is the factor coming from $P_{n+1}/P_{n-j}$, whose relative error decays like $\theta^{n-j}$. This produces the constant multiplying $\theta^{n-j}$ in the claim.

$$|\frac{1}{\pi_{n,j}} - \frac{p^{j+1}}{P_{j-1}}| = |\frac{p^{j+1}}{P_{j-1}}\frac{1+ \epsilon_{n+1}}{n-j}-\frac{p^{j+1}}{P_{j-1}}|$$
$$|\frac{1}{\pi_{n,j}} - \frac{p^{j+1}}{P_{j-1}}| \leq \frac{p^4 \kappa (1+\theta^2)}{1-\kappa \theta^3} \theta^{n-j}$$

The next cell evaluates this constant numerically (it should be approximately $3.704$), and also prints the sum of the left and right constants.

In [5]:
bound_right_side = mp.fdiv(mp.fmul(mp.power(p,4),mp.fmul(kappa,mp.fadd(1,mp.power(theta,2)))),mp.fsub(1,mp.fmul(kappa,mp.power(theta,3))))
print(bound_right_side)

print(bound_right_side + bound_left_side)


bound_central = mp.fdiv(mp.fadd(mp.fadd(mp.fmul(2,kappa), mp.fmul(mp.power(kappa,2),mp.power(theta,3))), mp.fmul(kappa,mp.power(theta,5))), mp.fmul(rho, mp.power(mp.fsub(1, mp.fmul(kappa,mp.power(theta,3))),2)))
print(mp.fsub(5.848, bound_central))

3.7031423793715103764045507584485972140765464906101
5.7464181911591936851421066702721053720238457627774
0.00058634635716815041106558299128817167155758658389662


## Second check: the series bound

Second check, only three things remain to be proved:

**(i) Left-boundary term.** Since $L_{2r-1}=L_{2r}=r$ and $0<\theta<1$, we can bound the finite sum by the infinite one and group the terms in pairs:

$$\sum_{j=1}^{m}L_j\theta^{j-1}\leq\sum_{j=1}^{\infty}L_j\theta^{j-1}=\sum_{r=1}^{\infty}r\left(\theta^{2r-2}+\theta^{2r-1}\right)=(1+\theta)\sum_{r=1}^{\infty}r\,\theta^{2r-2}=\frac{1+\theta}{(1-\theta^2)^2},$$

where the last equality uses $\sum_{r\geq1} r\,x^{r-1}=\frac{1}{(1-x)^2}$ with $x=\theta^2$. Therefore

$$\frac{(2.044)(18)}{n+6} \sum_{j=1}^{m}L_j \theta^{j-1} \leq \frac{(2.044)(18)}{n+6} \frac{1+\theta}{(1-\theta^2)^2} \leq \frac{188}{n+6},$$

because numerically $(2.044)(18)\,\dfrac{1+\theta}{(1-\theta^2)^2}\approx 187.63<188$.

**(ii) Right-boundary term.** For $j\leq m$ we have $L_j\leq\frac{j+1}{2}\leq\frac{m+1}{2}$, and the remaining sum is geometric:

$$\sum_{j=1}^{m}\theta^{n-j}=\theta^{n-m}\sum_{i=0}^{m-1}\theta^{i}\leq\frac{\theta^{n-m}}{1-\theta}.$$

Moreover $\frac{m+1}{n+6}\leq\frac{n+2}{2(n+6)}\leq\frac12$ and $n-m\geq m$, so $\theta^{n-m}\leq\theta^{m}$. Combining these facts,

$$\frac{(3.704)(18)}{n+6} \sum_{j=1}^{m}L_j \theta^{n-j} \leq \frac{(3.704)(18)}{n+6}\cdot\frac{m+1}{2}\cdot\frac{\theta^{n-m}}{1-\theta} \leq \frac{9(3.704)}{2(1-\theta)} \theta^{m} \leq 48.5 \theta^m,$$

since numerically $\dfrac{9(3.704)}{2(1-\theta)}\approx 48.43<48.5$.

**(iii) Central term ($n=2m+1$ odd).** Here $L_{m+1}=\lfloor\frac{m+2}{2}\rfloor\leq\frac{m+2}{2}$ and $n+6=2m+7$. Since $4(m+2)<2(2m+7)$, we get $\frac{m+2}{2(2m+7)}<\frac14$, hence

$$\frac{9 L_{m+1}}{n+6}(5.848)\theta^{m} \leq \frac{9(5.848)}{4}\theta^m \leq 13.2 \theta^m,$$

since numerically $\dfrac{9(5.848)}{4}\approx 13.16<13.2$. For $n$ even there is no central site, so this term is simply absent.

**Conclusion.** Adding the three contributions, and using $48.5+13.2=61.7<62$,

$$\Big|\sum_{j=1}^{n} g_{n,j}\,\Delta a_{n,k}(j)\Big| \leq \frac{188}{n+6} + (48.5+13.2)\,\theta^{m} < \frac{188}{n+6} + 62\,\theta^{\lfloor n/2\rfloor}.$$

In [6]:
comprobation_1 = mp.fdiv(mp.fmul(2.044,mp.fmul(18,mp.fadd(1,theta))),mp.power(mp.fsub(1,mp.power(theta,2)),2))
comprobation_2= mp.fdiv(mp.fmul(9,3.704),mp.fmul(2,mp.fsub(1,theta)))
comprobation_3 = mp.fdiv(mp.fmul(9,5.848),4)

print(comprobation_1)
print(comprobation_2)
print(comprobation_3)



187.61699641732949368282904792176342626473509363481
48.434567647196188232500993038549891133297839769588
13.157999999999999696242980462557170540094375610352


## Final check: the threshold $n \geq 73$

Finally, we check that the lower bound of the proposition is positive from $n=73$ on. We use that $\frac1\rho$ is larger than $\frac{12}{5}$, and that the remaining terms are decreasing in $n$, so it suffices to evaluate them at $n=73$, where $\lfloor n/2\rfloor = 36$ and $n+6=79$. The two printed numbers in the next cell are the differences with respect to $\frac{12}{5}$ and $\frac{1}{50}$ respectively, so both should be positive.

Final check:

$$\frac{1}{\rho}  > \frac{12}{5}$$


$$\frac{1}{\rho}-\frac{188}{n+6}- 62\theta^{\lfloor n/2 \rfloor} > \frac{1}{\rho}-\frac{188}{79}- 62\theta^{36} > \frac{1}{50}$$

In [7]:
first = mp.fdiv(1,rho)

print(mp.fsub(first,12/5))

second = first - mp.fdiv(188,79) - mp.fmul(62,mp.power(theta,36))
print(mp.fsub(second,1/50))

0.03015970900194682290644201189295458505073600813668
0.030397107366453099013276666585211697958544554339983
